# [9665] NLTK 2
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Amazon_product_reviews.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/09/25 15:08:48


### Import libraries

In [ ]:
import pandas as pd
import string
import nltk
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import LabelEncoder

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

### Load data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Amazon_product_reviews.csv')

### Examine data

In [ ]:
df.shape

(81, 7)

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df.sample(5)

,id,brand,manufacturer,name,customer_rating,customer_review,review_title
38,AVpfpK8KLJeJML43BCuD,Amazon,Amazon,Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker,1.0,"Amazon Tap is a great device. The device has very good audio quality despite it's small size. The biggest difference between this and the larger Amazon Echo, is that you have to press the ""microphone"" button before the device will listen to you (which is great for people who are concerned that the Echo is always listening to what you say). So far, the battery life has been great. With the Alexa app, it is very easy to connect this device to different Wi-Fi networks.",Amazon Tap is great!
42,AVph_N4z1cnluZ0-IkrF,Moshi,Amazon,Moshi Anti-Glare No Bubble Screen Protector for the Fire Phone,0.0,"UPDATE: July 6, 2015I'm reducing my rating to one star because all of a sudden this morning the screen protector just quit adhering to the screen. It falls off completely and if you do get it in place within five minutes it's slipping again and huge bubbles form. I have never had a problem with a Moshi product before and I am very surprised by this. At this price there is no excuse for the poor longevity of this product and I would never ever buy one again nor would I advise anybody to spend the money on this product to begin with.Original review follows in all its irony except for the whining about the price at the time that I still stand by:Superb screen protector. The best you can buy. Ridiculous price. Moshi products have been the stress reliever I've relied on when it comes to tablets and phone screen saver installation with their bubble proof installation and performance, but in this particular case I think the price I paid was twice what it was worth. So now the stress is back.",Updated Downgrade
34,AVpjWh8e1cnluZ0-Vy0O,Amazon,NaN,Fire HD 6 Tablet,1.0,"If you read my Fire TV review you know that I am tough on Amazon when it comes to their own items. It needs to deliver quality for the price point to earn stars from me. Please take the time to read my entire review and feel free to ask questions. I will do my best to respond to them as I can and update the review to reflect those answers and other things I discover along the way.First my background. I own many Amazon Kindles (bw, Fire gen2, Fire gen3) as well as Apple Ipad (gen 4), Samsung Note 3 and have an LG G2 smartphone (had a Samsung S4 before that), notebooks, chromebooks, etc. I have also used many other products including the Fire HDX line. I have a solid computer background as well but honestly I am more of a casual user when it comes to tablets like this one.Amazon has changed many things over the life of the Fire product line. Adding and removing features (like cameras--the first generation had one but the second generation removed them but not they are back).They are all useful devices but each one has its own niche so to speak so make sure whatever you get has the features that are important to you.Now to this model. This is like a big smartphone--which is a big difference from past kindles. It is more compact, lighter and yet still very usable. In fact for me, the shrinking of size increases the usefulness. I find that I use my smartphone way more than my tablets--even the smaller Fire HD and Samsung Note tablets so I am expecting this to replace more of that on the couch usage which is great since it so much less expensive than most smartphones. That means you can feel less bad about scratches that might occur or the accidental drops. (I am guilty of both.Read more","The single best value Fire in the entire line. Skip the Fire 7 and get the Fire 6 instead! 11,"
55,AVzRkFTFvKc47QAVd43-,Amazon,NaN,All-New Amazon Fire HD 8 Tablet Case (7th Generation,0.0,This is a horrible an overpriced case. I've had this case for less than two weeks and it's already coming apart. The little rubber stitching around the case is falling off,Don't waste your money on this
67,AVpfODXeilAPnD_xXeUd,Amazon,NaN,Replacement Rem

In [ ]:
df.isnull().sum()

,0
id,0
brand,0
manufacturer,43
name,0
customer_rating,0
customer_review,0
review_title,0


### Prepare data

In [ ]:
# Drop rows where customer_review is null
df.dropna(subset=['customer_review'], inplace=True)

In [ ]:
# Lowercase data
df['customer_review_clean'] = df['customer_review'].str.lower()

# Remove punctuation
translator = str.maketrans('', '', string.punctuation)
df['customer_review_clean'] = df['customer_review_clean'].str.translate(translator)

# Remove extra spaces
df['customer_review_clean'] = df['customer_review_clean'].str.replace('\s+', ' ')

In [ ]:
# Remove stop words
sw = set(stopwords.words("english"))
df['customer_review_clean'] = df['customer_review_clean'].apply(lambda x: " ".join(x for x in str(x).split() if x not in sw))

In [ ]:
# Remove rare words
#  First calculate words counts
temp_df = pd.Series(' '.join(df['customer_review_clean']).split()).value_counts()
temp_df

,count
amazon,59
fire,44
device,39
one,37
tap,36
...,...
perhaps,1
purchasefirst,1
heavier,1
imagined,1


In [ ]:
# Choose words with less than 2 frequencies to drop
rw = temp_df[temp_df <= 1]
rw

,count
2013,1
compare,1
gift,1
refurbish,1
defect,1
...,...
perhaps,1
purchasefirst,1
heavier,1
imagined,1


In [ ]:
# Remove rare words
df['customer_review_clean'] = df['customer_review_clean'].apply(lambda x: " ".join(x for x in x.split() if x not in rw))

### Text analysis operations with NLTK

In [ ]:
text = df['customer_review'][0]
text

"Let me guess: you love books, but you're not sure you want to get a kindle because you love the feel of books, rightI'm here to tell you that the kindle is the perfect balance of book and digital format.SHORT REVIEWYes, you should buy a kindle. Get the paperwhite with no ads. You're welcome.LONG REVIEWI love physical books too, I'm with you. But I know myself, and I know that once I forget to take the book I'm reading with me, that's it. I'll start another book and rarely finish the first. I also know if I try and read on my phone or iPad that I'll get distracted and start wondering about what's happening on the internet (Instagram's not gonna scroll ITSELF). Either way I'm not finishing the book.WHY KINDLEThe kindle takes the best of both worlds and mashes them together. The e ink display is honestly incredible. I wish iPhones had an e ink display. It really looks just like a printed page. So you get the experience of reading a physical paper book, but with the perks of being digital

In [ ]:
# Sentence tokenizer breaks text paragraph into sentences
text_tokenized_by_sent=sent_tokenize(text)
text_tokenized_by_sent

["Let me guess: you love books, but you're not sure you want to get a kindle because you love the feel of books, rightI'm here to tell you that the kindle is the perfect balance of book and digital format.SHORT REVIEWYes, you should buy a kindle.",
 'Get the paperwhite with no ads.',
 "You're welcome.LONG REVIEWI love physical books too, I'm with you.",
 "But I know myself, and I know that once I forget to take the book I'm reading with me, that's it.",
 "I'll start another book and rarely finish the first.",
 "I also know if I try and read on my phone or iPad that I'll get distracted and start wondering about what's happening on the internet (Instagram's not gonna scroll ITSELF).",
 "Either way I'm not finishing the book.WHY KINDLEThe kindle takes the best of both worlds and mashes them together.",
 'The e ink display is honestly incredible.',
 'I wish iPhones had an e ink display.',
 'It really looks just like a printed page.',
 "So you get the experience of reading a physical paper 

In [ ]:
# Word tokenizer breaks text paragraph into words
text_tokenized_by_word=word_tokenize(text)
print(text_tokenized_by_word)

['Let', 'me', 'guess', ':', 'you', 'love', 'books', ',', 'but', 'you', "'re", 'not', 'sure', 'you', 'want', 'to', 'get', 'a', 'kindle', 'because', 'you', 'love', 'the', 'feel', 'of', 'books', ',', 'rightI', "'m", 'here', 'to', 'tell', 'you', 'that', 'the', 'kindle', 'is', 'the', 'perfect', 'balance', 'of', 'book', 'and', 'digital', 'format.SHORT', 'REVIEWYes', ',', 'you', 'should', 'buy', 'a', 'kindle', '.', 'Get', 'the', 'paperwhite', 'with', 'no', 'ads', '.', 'You', "'re", 'welcome.LONG', 'REVIEWI', 'love', 'physical', 'books', 'too', ',', 'I', "'m", 'with', 'you', '.', 'But', 'I', 'know', 'myself', ',', 'and', 'I', 'know', 'that', 'once', 'I', 'forget', 'to', 'take', 'the', 'book', 'I', "'m", 'reading', 'with', 'me', ',', 'that', "'s", 'it', '.', 'I', "'ll", 'start', 'another', 'book', 'and', 'rarely', 'finish', 'the', 'first', '.', 'I', 'also', 'know', 'if', 'I', 'try', 'and', 'read', 'on', 'my', 'phone', 'or', 'iPad', 'that', 'I', "'ll", 'get', 'distracted', 'and', 'start', 'wonde

#### Stemming and lemmatization

In [ ]:
ps = PorterStemmer()
lem = WordNetLemmatizer()

word = "flying"
print("Stemmed Word:", ps.stem(word))
print("Lemmatized Word:", lem.lemmatize(word, "v"))
print("Lemmatized Word:", lem.lemmatize(word, "n"))

Stemmed Word: fli
Lemmatized Word: fly
Lemmatized Word: flying


#### POS Tagging

In [ ]:
nltk.pos_tag(text_tokenized_by_word)

[('Let', 'VB'),
 ('me', 'PRP'),
 ('guess', 'VB'),
 (':', ':'),
 ('you', 'PRP'),
 ('love', 'VBP'),
 ('books', 'NNS'),
 (',', ','),
 ('but', 'CC'),
 ('you', 'PRP'),
 ("'re", 'VBP'),
 ('not', 'RB'),
 ('sure', 'JJ'),
 ('you', 'PRP'),
 ('want', 'VBP'),
 ('to', 'TO'),
 ('get', 'VB'),
 ('a', 'DT'),
 ('kindle', 'NN'),
 ('because', 'IN'),
 ('you', 'PRP'),
 ('love', 'VBP'),
 ('the', 'DT'),
 ('feel', 'NN'),
 ('of', 'IN'),
 ('books', 'NNS'),
 (',', ','),
 ('rightI', 'NN'),
 ("'m", 'VBP'),
 ('here', 'RB'),
 ('to', 'TO'),
 ('tell', 'VB'),
 ('you', 'PRP'),
 ('that', 'IN'),
 ('the', 'DT'),
 ('kindle', 'NN'),
 ('is', 'VBZ'),
 ('the', 'DT'),
 ('perfect', 'JJ'),
 ('balance', 'NN'),
 ('of', 'IN'),
 ('book', 'NN'),
 ('and', 'CC'),
 ('digital', 'JJ'),
 ('format.SHORT', 'NN'),
 ('REVIEWYes', 'NNP'),
 (',', ','),
 ('you', 'PRP'),
 ('should', 'MD'),
 ('buy', 'VB'),
 ('a', 'DT'),
 ('kindle', 'NN'),
 ('.', '.'),
 ('Get', 'VB'),
 ('the', 'DT'),
 ('paperwhite', 'NN'),
 ('with', 'IN'),
 ('no', 'DT'),
 ('ads', 'NNS'

### Sentiment analysis

In [ ]:
# Instantiate Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

In [ ]:
# Get the lexicon
vader_lexicon = sia.lexicon
len(vader_lexicon)

7502

#### Review VADER lexicon

In [ ]:
# Display the 10 words and their sentiment scores
list(vader_lexicon.items())[500:510]

[('aches', -1.0),
 ('achievable', 1.3),
 ('aching', -2.2),
 ('acquit', 0.8),
 ('acquits', 0.1),
 ('acquitted', 1.0),
 ('acquitting', 1.3),
 ('acrimonious', -1.7),
 ('active', 1.7),
 ('actively', 1.3)]

In [ ]:
# Find words in the lexicon containing 'happy'
happy_words = {word: score for word, score in vader_lexicon.items() if 'happy' in word}
happy_words

{'happy': 2.7, 'unhappy': -1.8}

In [ ]:
# Display words with strong positive sentiment
positive_words = {word: score for word, score in vader_lexicon.items() if score > 2.0}
list(positive_words.items())[500:510]

[('hopeful', 2.3),
 ('hug', 2.1),
 ('hugs', 2.2),
 ('humoring', 2.1),
 ('humorously', 2.3),
 ('humorousness', 2.4),
 ('humour', 2.1),
 ('hurrah', 2.6),
 ('hurrahing', 2.4),
 ('hurrahs', 2.1)]

In [ ]:
# Display words with strong negative sentiment
negative_words = {word: score for word, score in vader_lexicon.items() if score < -2.0}
list(negative_words.items())[500:510]

[('hatred', -3.2),
 ('haunted', -2.1),
 ('havoc', -2.9),
 ('heartbreak', -2.7),
 ('heartbreaker', -2.2),
 ('heartbreakers', -2.1),
 ('heartbroken', -3.3),
 ('heartless', -2.2),
 ('heartlessly', -2.8),
 ('heartlessness', -2.8)]

#### Review VADER sentiment analysis scores for sample sentences

In [ ]:
sia.polarity_scores("I don't love this movie")

{'neg': 0.529, 'neu': 0.471, 'pos': 0.0, 'compound': -0.5216}

In [ ]:
sia.polarity_scores("I love this movie")

{'neg': 0.0, 'neu': 0.323, 'pos': 0.677, 'compound': 0.6369}

In [ ]:
sia.polarity_scores("I LOVE this movie")

{'neg': 0.0, 'neu': 0.288, 'pos': 0.712, 'compound': 0.7125}

In [ ]:
sia.polarity_scores("I LOVE this movie!!!")

{'neg': 0.0, 'neu': 0.256, 'pos': 0.744, 'compound': 0.7788}

In [ ]:
sia.polarity_scores("This meal was OK")

{'neg': 0.0, 'neu': 0.506, 'pos': 0.494, 'compound': 0.4466}

In [ ]:
sia.polarity_scores("I hate this. It's terrible!")

{'neg': 0.78, 'neu': 0.22, 'pos': 0.0, 'compound': -0.7959}

In [ ]:
sia.polarity_scores("Not bad at all!")

{'neg': 0.0, 'neu': 0.488, 'pos': 0.512, 'compound': 0.484}

#### Generate VADER sentiment analysis scores for dataframe

In [ ]:
# Calculate polarity scores for first 10 rows
df["customer_review_clean"][0:10].apply(lambda x: sia.polarity_scores(x))

,customer_review_clean
0,"{'neg': 0.073, 'neu': 0.613, 'pos': 0.314, 'compound': 0.9869}"
1,"{'neg': 0.229, 'neu': 0.693, 'pos': 0.079, 'compound': -0.4635}"
2,"{'neg': 0.157, 'neu': 0.667, 'pos': 0.177, 'compound': 0.451}"
3,"{'neg': 0.057, 'neu': 0.582, 'pos': 0.361, 'compound': 0.7751}"
4,"{'neg': 0.198, 'neu': 0.435, 'pos': 0.367, 'compound': 0.6456}"
5,"{'neg': 0.105, 'neu': 0.787, 'pos': 0.109, 'compound': 0.0258}"
6,"{'neg': 0.0, 'neu': 0.625, 'pos': 0.375, 'compound': 0.6369}"
7,"{'neg': 0.039, 'neu': 0.773, 'pos': 0.189, 'compound': 0.9373}"
8,"{'neg': 0.0, 'neu': 0.77, 'pos': 0.23, 'compound': 0.6697}"
9,"{'neg': 0.0, 'neu': 0.678, 'pos': 0.322, 'compound': 0.2263}"


In [ ]:
# Display only compound scores for first 10 rows
df["customer_review_clean"][0:10].apply(lambda x: sia.polarity_scores(x)["compound"])

,customer_review_clean
0,0.9869
1,-0.4635
2,0.4510
3,0.7751
4,0.6456
5,0.0258
6,0.6369
7,0.9373
8,0.6697
9,0.2263


In [ ]:
# Save compound polarity scores in a new column
df["polarity_score"] = df["customer_review_clean"].apply(lambda x: sia.polarity_scores(x)["compound"])
df[['customer_review', 'customer_review_clean', 'customer_rating', 'polarity_score']].head()

,customer_review,customer_review_clean,customer_rating,polarity_score
0,"Let me guess: you love books, but you're not sure you want to get a kindle because you love the feel of books, rightI'm here to tell you that the kindle is the perfect balance of book and digital format.SHORT REVIEWYes, you should buy a kindle. Get the paperwhite with no ads. You're welcome.LONG REVIEWI love physical books too, I'm with you. But I know myself, and I know that once I forget to take the book I'm reading with me, that's it. I'll start another book and rarely finish the first. I also know if I try and read on my phone or iPad that I'll get distracted and start wondering about what's happening on the internet (Instagram's not gonna scroll ITSELF). Either way I'm not finishing the book.WHY KINDLEThe kindle takes the best of both worlds and mashes them together. The e ink display is honestly incredible. I wish iPhones had an e ink display. It really looks just like a printed page. So you get the experience of reading a physical paper book, but with the perks of being digital.Namely:- Share what book you're reading to Goodreads, Facebook, or twitter (so you can look SMORT)- Built in dictionary (so you can learn the proper spelling of the word SMORT)- Export your highlights as a PDFPlus, it'll also sync with the kindle app on your phone so you can squeeze in the final few pages of the chapter while you're in the bathroom (don't pretend you don't do that. You're either on your phone or you're reading the febreeze ingredients)READING IN BEDThe backlight looks great. It's a perfect size.Read more",let guess love books youre sure want get kindle love feel books kindle perfect book digital buy kindle get ads youre love physical books im know know forget take book im reading thats ill start another book finish first also know try read phone ipad ill get distracted start wondering whats internet either way im kindle takes best e ink display honestly incredible wish e ink display really looks like page get experience reading physical paper book share book youre reading look smort learn word smort also kindle app phone pages youre dont dont youre either phone youre reading looks great perfect,1.0,0.9869
1,"Remote didn't work, did all the troubleshooting to connect, sent me a replacement remote that didn't fix it, and then when I sent the replacement remote, they couldn't find it and charged me anyway. Very disappointed.",remote didnt work troubleshooting connect sent replacement remote didnt sent replacement remote couldnt find charged anyway disappointed,0.0,-0.4635
2,"I purchase almost everything with the exception of food from Amazon. I have been a prime member since the inception of the program. Without exception, I have always received EXCELLENT customer service. Until now. I'm having difficulty with my Fire TV connecting to my wifi. Last Saturday I spent approximately 1- hours (maybe 2) troubleshooting with two women techs on the phone. Mary (who I was passed to from the first tech because of her expertise) said she thought it wasn't just a problem unique to me and asked me if she could call me back on Sunday after researching the problem. I had no problem with that. No call on Sunday so I called on Monday. Much to my surprise, there was no record of my call attached to my file. The rep was confused as to whether I was talking about fire wire or Fire TV so I was first sent to a different department. Finally, I got through that fire wire was not the same as Fire TV and I was transferred back to digital. We went through the same troubleshooting taking up a lot of my time. I was willing to do it until we were going around in circles and my phone was running out of juice. The gentleman asked me if he could research and call me later in the day. NO CALL. It's Tuesday--still no call.It really pains me to have to say anything negative about Amazon's Customer Service as it has NEVER BEEN LESS THAN EXCELLENT IN THE PAST. I've noticed I haven't gotten a survey

In [ ]:
# Use polarity score to generate labels
df["polarity_score"][0:10].apply(lambda x: "pos" if x > 0 else "neg")

,polarity_score
0,pos
1,neg
2,pos
3,pos
4,pos
5,pos
6,pos
7,pos
8,pos
9,pos


In [ ]:
# Save compound polarity scores as labels in a new column
df["polarity_label"] = df["polarity_score"].apply(lambda x: "pos" if x > 0 else "neg")
df[['customer_review_clean', 'customer_rating', 'polarity_score', 'polarity_label']].head(10)

,customer_review_clean,customer_rating,polarity_score,polarity_label
0,let guess love books youre sure want get kindle love feel books kindle perfect book digital buy kindle get ads youre love physical books im know know forget take book im reading thats ill start another book finish first also know try read phone ipad ill get distracted start wondering whats internet either way im kindle takes best e ink display honestly incredible wish e ink display really looks like page get experience reading physical paper book share book youre reading look smort learn word smort also kindle app phone pages youre dont dont youre either phone youre reading looks great perfect,1.0,0.9869,pos
1,remote didnt work troubleshooting connect sent replacement remote didnt sent replacement remote couldnt find charged anyway disappointed,0.0,-0.4635,neg
2,purchase almost exception amazon prime since without exception always received excellent customer service im difficulty fire tv connecting wifi last saturday spent 1 hours maybe 2 troubleshooting two techs phone first tech said thought wasnt problem asked could call back sunday problem problem call sunday called much surprise record call file whether talking fire wire fire tv first sent different finally got fire wire fire tv back digital went troubleshooting taking lot time going around phone asked could call day call really say anything negative amazons customer service never less excellent past ive havent usually customer service call whoever amazon digital customer service excellent customer service reputation lose im going record hours time phone trying problem record two call back would received call saying come hoping amazon kept record past reviews review account see much ive spent years customer im disappointed write review im hoping someone change things amazons reputation cant one kind people techs fire tv techs ive past fact dont think even,0.0,0.4510,pos
3,still really love new kindle returning case stands doesnt matter nothing complete waste money,0.0,0.7751,pos
4,small dont voice interface one like wasnt sure amazon fire tv would support one remote amazon fire device,1.0,0.6456,pos
5,complete waste read reviews thought product better thin piece plastic lasted one week husband user either someone cell uses daily probably last day,0.0,0.0258,pos
6,love fact battery issues listening amount bass size,1.0,0.6369,pos
7,taking time read device quite honestly although slightly smaller last years release device voice even better im talking talking better say alexa even im even im music thru speaker found last years echo dot device kitchen get listen second device try voice technology amazon awesome feature one echo dot command multiple devices dot word alexa side note basic amazon music unlimited work one devices one time one want use one dot echo purchase plan selling point device use exsisting bluetooth speakers amazon use 35mm jack back connect home really give risk course amazon tap echo unfortunately devices comes audio bluetooth 35mm jack device allow music thru exsisting,1.0,0.9373,pos
8,couldnt one buy things ended buying tap portable take coffee listen music put absolutely love husband,1.0,0.6697,pos
9,even speaker item worth price,1.0,0.2263,pos


In [ ]:
df["customer_rating"].value_counts()

,count
customer_rating,
0.0,42
1.0,39


In [ ]:
df["polarity_label"].value_counts()

,count
polarity_label,
pos,59
neg,22


In [ ]:
df["polarity_label"] = LabelEncoder().fit_transform(df["polarity_label"])
df[['customer_review_clean', 'customer_rating', 'polarity_score', 'polarity_label']].head(10)

,customer_review_clean,customer_rating,polarity_score,polarity_label
0,let guess love books youre sure want get kindle love feel books kindle perfect book digital buy kindle get ads youre love physical books im know know forget take book im reading thats ill start another book finish first also know try read phone ipad ill get distracted start wondering whats internet either way im kindle takes best e ink display honestly incredible wish e ink display really looks like page get experience reading physical paper book share book youre reading look smort learn word smort also kindle app phone pages youre dont dont youre either phone youre reading looks great perfect,1.0,0.9869,1
1,remote didnt work troubleshooting connect sent replacement remote didnt sent replacement remote couldnt find charged anyway disappointed,0.0,-0.4635,0
2,purchase almost exception amazon prime since without exception always received excellent customer service im difficulty fire tv connecting wifi last saturday spent 1 hours maybe 2 troubleshooting two techs phone first tech said thought wasnt problem asked could call back sunday problem problem call sunday called much surprise record call file whether talking fire wire fire tv first sent different finally got fire wire fire tv back digital went troubleshooting taking lot time going around phone asked could call day call really say anything negative amazons customer service never less excellent past ive havent usually customer service call whoever amazon digital customer service excellent customer service reputation lose im going record hours time phone trying problem record two call back would received call saying come hoping amazon kept record past reviews review account see much ive spent years customer im disappointed write review im hoping someone change things amazons reputation cant one kind people techs fire tv techs ive past fact dont think even,0.0,0.4510,1
3,still really love new kindle returning case stands doesnt matter nothing complete waste money,0.0,0.7751,1
4,small dont voice interface one like wasnt sure amazon fire tv would support one remote amazon fire device,1.0,0.6456,1
5,complete waste read reviews thought product better thin piece plastic lasted one week husband user either someone cell uses daily probably last day,0.0,0.0258,1
6,love fact battery issues listening amount bass size,1.0,0.6369,1
7,taking time read device quite honestly although slightly smaller last years release device voice even better im talking talking better say alexa even im even im music thru speaker found last years echo dot device kitchen get listen second device try voice technology amazon awesome feature one echo dot command multiple devices dot word alexa side note basic amazon music unlimited work one devices one time one want use one dot echo purchase plan selling point device use exsisting bluetooth speakers amazon use 35mm jack back connect home really give risk course amazon tap echo unfortunately devices comes audio bluetooth 35mm jack device allow music thru exsisting,1.0,0.9373,1
8,couldnt one buy things ended buying tap portable take coffee listen music put absolutely love husband,1.0,0.6697,1
9,even speaker item worth price,1.0,0.2263,1
